# 06B - Publishing a Pipeline — Azure ML SDK v2 Version

This notebook is a modern SDK v2 replacement for the old SDK v1 **Publishing a Pipeline** lab.

The old lab used SDK v1 objects such as:

- `Workspace`
- `PublishedPipeline`
- pipeline REST endpoints
- `InteractiveLoginAuthentication`
- `requests.post(...)`
- `PipelineRun`
- `ScheduleRecurrence`
- `Schedule`

Those patterns belong to Azure ML SDK v1. SDK v1 is deprecated, and in SDK v2 the equivalent architecture is different.

## Modern SDK v2 replacement

| Old SDK v1 idea | SDK v2 replacement |
|---|---|
| Published Pipeline | Pipeline component deployment |
| Pipeline endpoint | Batch endpoint |
| Published pipeline version | Batch deployment |
| REST call to published pipeline | Invoke batch endpoint |
| Pipeline schedule from published pipeline ID | `JobSchedule` using a pipeline job |
| `RunDetails` widget | Studio URL + `ml_client.jobs.stream()` |

This lab does three things:

1. Rebuilds the diabetes training pipeline from SDK v2.
2. Deploys the pipeline component behind a batch endpoint.
3. Shows how to invoke it and schedule the pipeline.

> Important: This notebook is intentionally self-contained. It does **not** depend on the previous 06A notebook having been run.

## 1. Connect to the Azure ML workspace

This uses `MLClient.from_config()`, which works inside Azure ML Studio notebooks when your workspace config is available.

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
ml_client = MLClient.from_config(credential=credential)

print("Connected to workspace:", ml_client.workspace_name)
print("Resource group:", ml_client.resource_group_name)
print("Subscription:", ml_client.subscription_id)

## 2. Check that local data files exist

This lab expects the diabetes CSV files to be available in the local `./data` folder.

Required files:

- `./data/diabetes.csv`
- `./data/diabetes2.csv`

In [ ]:
from pathlib import Path

DATA_FOLDER = Path("./data")
expected_files = [DATA_FOLDER / "diabetes.csv", DATA_FOLDER / "diabetes2.csv"]

print("Current folder:", Path.cwd())
print("Data folder exists:", DATA_FOLDER.exists())

if DATA_FOLDER.exists():
    print("Files in ./data:")
    for item in DATA_FOLDER.iterdir():
        print(" -", item)

missing = [str(p) for p in expected_files if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required data files: " + ", ".join(missing) + "\n"
        "Upload the diabetes CSV files into a ./data folder before running this lab."
    )

print("Data files found.")

## 3. Create or update the diabetes Data asset

SDK v1 used `Dataset`. SDK v2 uses `Data` assets.

We upload the entire `./data` folder as a `uri_folder`.

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
from datetime import datetime

DATA_ASSET_NAME = "diabetes-dataset"
data_version = datetime.now().strftime("%Y%m%d%H%M%S")

diabetes_data = Data(
    name=DATA_ASSET_NAME,
    version=data_version,
    type=AssetTypes.URI_FOLDER,
    path=str(DATA_FOLDER),
    description="Diabetes CSV files uploaded from local ./data folder for SDK v2 publishing pipeline lab",
    tags={"format": "CSV", "lab": "06B publishing pipeline SDK v2"},
)

diabetes_data = ml_client.data.create_or_update(diabetes_data)

print("Data asset created or updated.")
print("Name:", diabetes_data.name)
print("Version:", diabetes_data.version)
print("Path:", diabetes_data.path)

## 4. Create the source folder

This folder stores the training script and Conda environment file used by the pipeline component.

In [ ]:
from pathlib import Path

source_folder = Path("diabetes_publish_pipeline_v2")
source_folder.mkdir(exist_ok=True)

print("Source folder:", source_folder.resolve())

## 5. Create the training script

This script trains a decision-tree model and writes outputs to an Azure ML output folder.

Notice the important stability choice:

- We do **not** use `mlflow.log_figure()`.
- That method can break when MLflow and Azure ML artifact plugins are mismatched.
- Instead, we save the ROC chart directly into the output folder.

In [ ]:
%%writefile $source_folder/train_diabetes.py
import argparse
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier


def load_csv_folder(folder_path: str) -> pd.DataFrame:
    folder = Path(folder_path)
    csv_files = sorted(folder.glob("*.csv"))

    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in: {folder}")

    frames = []
    for csv_file in csv_files:
        print(f"Loading: {csv_file}")
        frames.append(pd.read_csv(csv_file))

    return pd.concat(frames, ignore_index=True)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--training_data", type=str, required=True)
    parser.add_argument("--output_model", type=str, required=True)
    args = parser.parse_args()

    output_dir = Path(args.output_model)
    output_dir.mkdir(parents=True, exist_ok=True)

    print("Training data path:", args.training_data)
    print("Output model path:", output_dir)

    diabetes = load_csv_folder(args.training_data)

    feature_columns = [
        "Pregnancies",
        "PlasmaGlucose",
        "DiastolicBloodPressure",
        "TricepsThickness",
        "SerumInsulin",
        "BMI",
        "DiabetesPedigree",
        "Age",
    ]

    X = diabetes[feature_columns].values
    y = diabetes["Diabetic"].values

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=0,
    )

    print("Training decision-tree model...")
    model = DecisionTreeClassifier(random_state=0)
    model.fit(X_train, y_train)

    y_hat = model.predict(X_test)
    accuracy = float(np.average(y_hat == y_test))

    y_scores = model.predict_proba(X_test)
    auc = float(roc_auc_score(y_test, y_scores[:, 1]))

    print("Accuracy:", accuracy)
    print("AUC:", auc)

    metrics_path = output_dir / "metrics.json"
    with metrics_path.open("w") as f:
        json.dump({"accuracy": accuracy, "auc": auc}, f, indent=2)

    model_path = output_dir / "diabetes_model.pkl"
    joblib.dump(model, model_path)

    fpr, tpr, _ = roc_curve(y_test, y_scores[:, 1])
    fig = plt.figure(figsize=(6, 4))
    plt.plot([0, 1], [0, 1], "k--")
    plt.plot(fpr, tpr)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve - Decision Tree")
    plt.tight_layout()

    roc_path = output_dir / "roc_curve_decision_tree.png"
    fig.savefig(roc_path, bbox_inches="tight")
    plt.close(fig)

    print("Saved model:", model_path)
    print("Saved metrics:", metrics_path)
    print("Saved ROC chart:", roc_path)


if __name__ == "__main__":
    main()

## 6. Create or reuse compute

This lab uses an Azure ML compute cluster named `aml-cluster`.

If it does not exist, this cell creates it.

In [ ]:
from azure.ai.ml.entities import AmlCompute

cluster_name = "aml-cluster"

try:
    compute = ml_client.compute.get(cluster_name)
    print(f"Found existing compute cluster: {cluster_name}")
    print("Provisioning state:", compute.provisioning_state)
except Exception:
    print(f"Compute cluster {cluster_name} not found. Creating it now...")

    compute = AmlCompute(
        name=cluster_name,
        size="STANDARD_D2_V2",
        min_instances=0,
        max_instances=4,
        idle_time_before_scale_down=120,
    )

    compute = ml_client.compute.begin_create_or_update(compute).result()
    print("Compute created:", compute.name)
    print("Provisioning state:", compute.provisioning_state)

## 7. Create the environment

This replaces the SDK v1 `RunConfiguration` / `CondaDependencies` pattern.

In [ ]:
%%writefile diabetes_publish_pipeline_v2/conda.yml
name: diabetes-publish-pipeline-env
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
      - pandas
      - numpy
      - scikit-learn
      - matplotlib
      - joblib

In [ ]:
from azure.ai.ml.entities import Environment

pipeline_env = Environment(
    name="diabetes-publish-pipeline-env",
    description="Environment for publishing a diabetes training pipeline with SDK v2",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
    conda_file=str(source_folder / "conda.yml"),
)

pipeline_env = ml_client.environments.create_or_update(pipeline_env)
env_ref = f"{pipeline_env.name}:{pipeline_env.version}"

print("Environment registered:", env_ref)

## 8. Define the command component

SDK v1 published pipelines built from steps. SDK v2 publishes/deploys reusable components.

This command component is the training step.

In [ ]:
from azure.ai.ml import Input, Output, command
from azure.ai.ml.constants import AssetTypes

train_component = command(
    name="train_diabetes_publish_component",
    display_name="Train Diabetes Model for Pipeline Publishing Lab",
    description="Train a decision-tree model from diabetes CSV files.",
    code=str(source_folder),
    command=(
        "python train_diabetes.py "
        "--training_data ${{inputs.training_data}} "
        "--output_model ${{outputs.output_model}}"
    ),
    inputs={
        "training_data": Input(type=AssetTypes.URI_FOLDER),
    },
    outputs={
        "output_model": Output(type=AssetTypes.URI_FOLDER),
    },
    environment=env_ref,
)

print("Training command component defined.")

## 9. Build the pipeline

This is the pipeline that will later become a pipeline component deployment behind a batch endpoint.

In [ ]:
from azure.ai.ml import dsl

@dsl.pipeline(
    name="diabetes_training_publish_pipeline_v2",
    description="SDK v2 pipeline used to demonstrate publishing/deploying a pipeline.",
)
def diabetes_training_publish_pipeline(training_data: Input(type=AssetTypes.URI_FOLDER)):
    train_job = train_component(training_data=training_data)
    return {
        "trained_model": train_job.outputs.output_model
    }

print("Pipeline function created.")

## 10. Test the pipeline once

This is the SDK v2 equivalent of running the pipeline before publishing it.

Do not skip this step. If this test run fails, publishing the pipeline just hides the real problem behind an endpoint.

In [ ]:
pipeline_input = Input(
    type=AssetTypes.URI_FOLDER,
    path=f"azureml:{diabetes_data.name}:{diabetes_data.version}",
)

test_pipeline_job = diabetes_training_publish_pipeline(training_data=pipeline_input)
test_pipeline_job.settings.default_compute = cluster_name

returned_test_job = ml_client.jobs.create_or_update(
    test_pipeline_job,
    experiment_name="diabetes-training-pipeline-v2"
)

print("Pipeline test job submitted.")
print("Job name:", returned_test_job.name)
print("Status:", returned_test_job.status)
print("Studio URL:", returned_test_job.studio_url)

In [ ]:
# Stream logs and wait for completion.
# If this cell fails, open the Studio URL from the previous cell and inspect child-job logs.

ml_client.jobs.stream(returned_test_job.name)

completed_test_job = ml_client.jobs.get(returned_test_job.name)

print("Status:", completed_test_job.status)
print("Studio URL:", completed_test_job.studio_url)

if completed_test_job.status != "Completed":
    raise RuntimeError(
        f"Pipeline test job did not complete successfully. Status: {completed_test_job.status}."
    )

## 11. Convert the pipeline into a pipeline component

This is the key SDK v2 replacement for SDK v1 publishing.

In SDK v1, you published a pipeline run.  
In SDK v2, you deploy a **pipeline component** behind a **batch endpoint**.

In [ ]:
# Create and register a pipeline component from the pipeline definition.

pipeline_component = diabetes_training_publish_pipeline().component
registered_pipeline_component = ml_client.components.create_or_update(pipeline_component)

print("Pipeline component registered.")
print("Name:", registered_pipeline_component.name)
print("Version:", registered_pipeline_component.version)

## 12. Create a batch endpoint

A batch endpoint is the SDK v2 replacement for the old SDK v1 pipeline endpoint.

**Important naming rule:** Azure ML endpoint names must be **3–32 characters** and can only use safe lowercase letters, numbers, and hyphens.

The earlier generated name `diabetes-training-pipeline-endpoint` was too long, so this version creates a shorter timestamped name automatically.


In [ ]:
from azure.ai.ml.entities import BatchEndpoint
from datetime import datetime

# Azure ML endpoint names must be 3–32 characters, lowercase letters/numbers/hyphens,
# and should start/end with a letter or number.
# The previous name was too long: diabetes-training-pipeline-endpoint = 35 chars.
endpoint_name = f"diabetes-pipe-{datetime.now().strftime('%m%d%H%M')}"

print("Endpoint name:", endpoint_name)
print("Endpoint name length:", len(endpoint_name))

endpoint = BatchEndpoint(
    name=endpoint_name,
    description="Batch endpoint that invokes the diabetes training pipeline component.",
    tags={"lab": "06B", "type": "pipeline-component-endpoint"},
)

endpoint = ml_client.batch_endpoints.begin_create_or_update(endpoint).result()

print("Batch endpoint ready.")
print("Name:", endpoint.name)
print("Scoring URI:", getattr(endpoint, "scoring_uri", None))


## 13. Create a pipeline component deployment

This is the SDK v2 equivalent of a published pipeline version.

The endpoint is the stable interface.  
The deployment is the implementation behind that endpoint.

In [ ]:
from azure.ai.ml.entities import PipelineComponentBatchDeployment

deployment_name = "diabetes-training-v1"

deployment = PipelineComponentBatchDeployment(
    name=deployment_name,
    endpoint_name=endpoint_name,
    component=registered_pipeline_component,
    description="Deployment v1 of the diabetes training pipeline component.",
    settings={
        "continue_on_step_failure": False,
        "default_compute": cluster_name,
    },
)

deployment = ml_client.batch_deployments.begin_create_or_update(deployment).result()

print("Pipeline component deployment ready.")
print("Endpoint:", endpoint_name)
print("Deployment:", deployment.name)

## 14. Set the deployment as the default

When you invoke the endpoint without specifying a deployment, Azure ML uses the default deployment.

In [ ]:
endpoint = ml_client.batch_endpoints.get(endpoint_name)
endpoint.defaults.deployment_name = deployment_name

endpoint = ml_client.batch_endpoints.begin_create_or_update(endpoint).result()

print("Default deployment set.")
print("Endpoint:", endpoint.name)
print("Default deployment:", endpoint.defaults.deployment_name)

## 15. Invoke the endpoint with SDK v2

This replaces the old SDK v1 `requests.post(rest_endpoint, ...)` pattern.

The endpoint run is asynchronous. Azure ML returns a job, and then you track that job.

In [ ]:
endpoint_job = ml_client.batch_endpoints.invoke(
    endpoint_name=endpoint_name,
    inputs={
        "training_data": Input(
            type=AssetTypes.URI_FOLDER,
            path=f"azureml:{diabetes_data.name}:{diabetes_data.version}",
        )
    },
)

print("Endpoint invoked.")
print("Returned batch job resource name:", endpoint_job.name)

# The object returned by batch_endpoints.invoke() is a BatchJobResource.
# It does not expose .status or .studio_url directly, so fetch the real job object.
submitted_endpoint_job = ml_client.jobs.get(endpoint_job.name)

print("Status:", submitted_endpoint_job.status)
print("Studio URL:", submitted_endpoint_job.studio_url)


In [ ]:
# Optional but recommended: stream the endpoint-created job.

ml_client.jobs.stream(endpoint_job.name)

completed_endpoint_job = ml_client.jobs.get(endpoint_job.name)

print("Status:", completed_endpoint_job.status)
print("Studio URL:", completed_endpoint_job.studio_url)

if completed_endpoint_job.status != "Completed":
    raise RuntimeError(
        f"Endpoint job did not complete successfully. Status: {completed_endpoint_job.status}. "
        "Open the Studio URL above and inspect the child-job logs."
    )


## 16. Get the endpoint invocation URL

If you need REST integration later, use the batch endpoint invocation URL.

For teaching labs, SDK invocation is cleaner and safer than manual REST token handling.

In [ ]:
batch_endpoint = ml_client.batch_endpoints.get(endpoint_name)

print("Endpoint name:", batch_endpoint.name)
print("Invocation URL:", getattr(batch_endpoint, "scoring_uri", None))

## 17. Create a weekly schedule for the pipeline job

SDK v2 can schedule a pipeline job directly. You do **not** need to publish the pipeline first.

This schedule is set for every Monday at 00:00 UTC.

> Cost warning: this creates a recurring Azure ML schedule. Disable it after the lab if you do not want future runs.

In [ ]:
from azure.ai.ml import load_job
from azure.ai.ml.entities import JobSchedule, RecurrenceTrigger, RecurrencePattern
from datetime import datetime
from pathlib import Path
import yaml

# Azure ML schedule names must also be unique in the workspace.
schedule_name = f"weekly-diabetes-{datetime.now():%m%d%H%M}"

# JobSchedule works most reliably with a loaded job definition, not with the live @dsl.pipeline builder object.
# So we first write a small pipeline-job YAML file, then load it with load_job().
pipeline_job_yaml = {
    "$schema": "https://azuremlschemas.azureedge.net/latest/pipelineJob.schema.json",
    "type": "pipeline",
    "display_name": "weekly-diabetes-training-template",
    "description": "Scheduled SDK v2 diabetes training pipeline job.",
    "settings": {
        "default_compute": cluster_name,
    },
    "inputs": {
        "training_data": {
            "type": "uri_folder",
            "path": f"azureml:{diabetes_data.name}:{diabetes_data.version}",
        }
    },
    "jobs": {
        "diabetes_training_pipeline_component": {
            "type": "pipeline",
            "component": f"azureml:{registered_pipeline_component.name}:{registered_pipeline_component.version}",
            "inputs": {
                "training_data": "${{parent.inputs.training_data}}"
            },
        }
    },
}

yaml_path = Path("diabetes_pipeline_job.yml")
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(pipeline_job_yaml, f, sort_keys=False)

print("Pipeline job YAML written to:", yaml_path.resolve())

scheduled_pipeline_job = load_job(str(yaml_path))

recurrence_trigger = RecurrenceTrigger(
    frequency="week",
    interval=1,
    schedule=RecurrencePattern(
        week_days=["monday"],
        hours=[0],
        minutes=[0],
    ),
    time_zone="UTC",
)

job_schedule = JobSchedule(
    name=schedule_name,
    display_name="Weekly Diabetes Training V2",
    description="Runs the diabetes training pipeline every Monday at 00:00 UTC.",
    trigger=recurrence_trigger,
    create_job=scheduled_pipeline_job,
)

created_schedule = ml_client.schedules.begin_create_or_update(
    schedule=job_schedule
).result()

print("Schedule created or updated.")
print("Name:", created_schedule.name)
print("Enabled:", created_schedule.is_enabled)


## 18. List schedules

In [ ]:
schedules = list(ml_client.schedules.list())

if not schedules:
    print("No schedules found.")
else:
    for schedule in schedules:
        print("Name:", schedule.name)
        print("Display name:", schedule.display_name)
        print("Enabled:", schedule.is_enabled)
        print("---")

## 19. Disable the schedule after the lab

Run this cell if you do not want the weekly schedule to keep triggering jobs.

This is included because student subscriptions and trial credits can be wasted by forgotten schedules.

In [ ]:
# Uncomment and run this cell when you want to disable the schedule.
# disabled_schedule = ml_client.schedules.begin_disable(name=schedule_name).result()
# print("Schedule disabled:", disabled_schedule.name)
# print("Enabled:", disabled_schedule.is_enabled)

## What changed from the old lab?

| Old SDK v1 object/pattern | SDK v2 replacement |
|---|---|
| `Workspace.from_config()` | `MLClient.from_config()` |
| `pipeline_run.publish_pipeline(...)` | Register pipeline component + deploy to batch endpoint |
| `published_pipeline.endpoint` | `BatchEndpoint.scoring_uri` |
| `InteractiveLoginAuthentication()` | `DefaultAzureCredential()` |
| `requests.post(...)` for first test | `ml_client.batch_endpoints.invoke(...)` |
| `PipelineRun(...)` + `RunDetails` widget | `ml_client.jobs.get(...)`, `ml_client.jobs.stream(...)`, Studio URL |
| `ScheduleRecurrence` / `Schedule` | `JobSchedule`, `RecurrenceTrigger`, `RecurrencePattern` |

## Important design note

SDK v2 did not just rename SDK v1 pipeline publishing.

The modern production shape is:

```text
Pipeline function
    ↓
Pipeline component
    ↓
Batch endpoint
    ↓
Pipeline component deployment
    ↓
Invoke endpoint / schedule job
```

That is why this notebook is not a simple line-by-line conversion. A line-by-line conversion would be fragile and outdated.